# Controlled prediction errors before rounding

Start with the **unrounded expected need** $\mu_i$ from the multiclass model and true need $h_i$:

$$z_i=h_i+\beta(\mu_i-h_i),\qquad H_i=\operatorname{round}(z_i/\delta).$$

The DP receives integer $H_i$ with scale $1/\delta$, representing need $\delta H_i$.
There is only one rounding step. NumPy rounds exact halfway values to the nearest even grid index.
Here $\beta$ controls the error **before rounding**, while $\delta$ controls the grid precision.
The old beta experiment rounds the prediction **before** shrinking its error; its saved results are separate.

- $\beta=0$ reproduces LDA-tru for every supported grid.
- $\beta=1,\delta=1$ reproduces the existing rounded LDA-exp.
- $\beta=1$ on a finer grid uses more precise predicted needs and need not match that baseline.
- Alpha is fixed at zero. Model probabilities, the existing Rashomon rule, and utility tolerance stay fixed.

Only LDA-exp runs here. The same cohorts and eligible models are used for all beta/grid combinations.
True needs are used as an **oracle diagnostic** to construct controlled errors; this is not a deployable predictor.
This notebook saves allocations and error diagnostics for later evaluation against true needs.


In [6]:
import hashlib
import importlib
import json
import sys
from ast import literal_eval
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/allocations.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research repository or its test/ folder.")
sys.path.insert(0, str(ROOT))
import src.allocations
importlib.reload(src.allocations)
from src.allocations import Allocations


## Settings

Defaults match the current experiment runner: n=250, selection rate=0.1, 100 candidate models,
and 10 cohorts per seed. For a quick pilot, reduce `N_MODELS` and `ITERS_PER_SEED` first;
reducing candidate models also changes the candidate Rashomon pool, so treat those runs as a pilot.
Pilot and full runs are automatically saved in different configuration folders.

Use `average` predictions: argmax predictions have already discarded the fractional expected needs.
Grid steps must be reciprocals of positive integers (1, 0.5, 0.2, and optionally 0.1).
Increasing precision can increase DP runtime substantially. Add 0.1 later to check convergence.


In [7]:
METHOD = "bootstrap"
MODEL_CLASS = "nn_multiclass"
PRED_METHOD = "average"
NEED_THRESHOLD = 1
NEED_COL = "gagne_sum_t"
EXP_NEED_COL = "exp_need_{model}"
GROUP_COL = "race"
RASHOMON_EPS = 0.01
ALLOCATION_EPS = 0.1
N_MODELS = 100
SAMPLE_SIZE = 100
SELECTION_RATE = 0.1
ITERS_PER_SEED = 5

BETAS = [0.0, 0.25, 0.5, 0.75, 1.0]
NEED_STEPS = [1.0, 0.5, 0.2]
SKIP_EXISTING = True

PREDICTION_PATH = ROOT / (
    f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_qualified_gagne_{NEED_THRESHOLD}.csv"
)
RAW_DATA_PATH = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
RESULTS_ROOT = ROOT / "data/results/allocations/error_before_rounding"


In [8]:
if PRED_METHOD != "average":
    raise ValueError("This experiment requires the saved unrounded average predictions.")
if not BETAS or not NEED_STEPS:
    raise ValueError("Specify at least one beta and grid step.")
if len(set(BETAS)) != len(BETAS) or len(set(NEED_STEPS)) != len(NEED_STEPS):
    raise ValueError("Duplicate beta or grid values would repeat a run.")
for beta in BETAS:
    Allocations._validate_need_experiment(0.0, beta, 0)
for step in NEED_STEPS:
    Allocations._need_grid_scale(0.0, "before_rounding", step)

predictions = pd.read_csv(PREDICTION_PATH)
groups = pd.read_csv(RAW_DATA_PATH, usecols=["person_id", GROUP_COL])
df = predictions.merge(groups, on="person_id", how="left", validate="many_to_one")
model_columns = [f"m_{i}" for i in range(1, N_MODELS + 1)]
need_columns = [Allocations.exp_need_col_for_model(model, EXP_NEED_COL) for model in model_columns]
required = ["person_id", "seed", "y", NEED_COL, GROUP_COL, *model_columns, *need_columns]
missing = [column for column in required if column not in df]
if missing:
    raise ValueError(f"Missing prediction columns: {missing}")
seeds = sorted(df["seed"].unique().tolist())
if seeds != list(range(len(seeds))):
    raise ValueError("The current cohort sampler expects consecutive seed IDs starting at zero.")
people_rows = df["person_id"].ge(0)
if not df.loc[people_rows, GROUP_COL].isin([0, 1]).all():
    raise ValueError("Every person must have a group label of 0 or 1.")
if df.loc[people_rows].duplicated(["seed", "person_id"]).any():
    raise ValueError("Person IDs must be unique within each seed.")

def make_allocator():
    return Allocations(
        df=df, sample_size=SAMPLE_SIZE, selection_rate=SELECTION_RATE,
        qualification_col="y", need_col=NEED_COL, group_col=GROUP_COL,
        rashomon_eps=RASHOMON_EPS, allocation_eps=ALLOCATION_EPS,
        n_models=N_MODELS, iters_per_seed=ITERS_PER_SEED,
    )

reference_allocator = make_allocator()
eligible_models = {
    int(seed): reference_allocator.rashomon_models(df.loc[(df["seed"] == seed) & (df["person_id"] < 0)])
    for seed in seeds
}
expected_diagnostic_keys = {
    (seed, iteration, model)
    for seed, models in eligible_models.items()
    for iteration in range(ITERS_PER_SEED)
    for model in models
}
print(f"n={SAMPLE_SIZE}, k={reference_allocator.k}; {len(seeds)} seeds, {ITERS_PER_SEED} cohorts per seed")
print("Eligible models by seed:", {seed: len(models) for seed, models in eligible_models.items()})
print(f"{len(BETAS) * len(NEED_STEPS)} LDA-exp runs; OPT and LDA-tru are not rerun.")


n=100, k=10; 5 seeds, 5 cohorts per seed
Eligible models by seed: {0: 100, 1: 100, 2: 100, 3: 100, 4: 98}
15 LDA-exp runs; OPT and LDA-tru are not rerun.


## Saving and resuming

Each configuration gets a separate directory under `data/results/allocations/error_before_rounding/`.
Its identity includes settings, input-file hashes, and the allocation-code hash. Existing alpha/beta
files are untouched. A completed beta/grid run contains allocations, cohort identities, error diagnostics,
and a metadata file recording runtime and output hashes. Rerunning skips verified completed runs;
an incomplete run is recomputed. Cohort identities are checked across the sweep.

Diagnostics have one row per seed, cohort, and eligible model:

| Prefix | Error measured |
|---|---|
| `prediction` | Original unrounded prediction minus true need |
| `before_rounding` | Adjusted need minus true need |
| `rounding` | Grid-rounded need minus adjusted need |
| `after_rounding` | Grid-rounded need minus true need |

MAE and RMSE are computed over the entire cohort. Before rounding they scale with beta;
after rounding they need not. These errors are not allocation disparity. Use the saved selections
and true needs to evaluate disparity in subsequent analysis. Runtime is measured for the whole run,
including DP, cohort sampling, and diagnostic computation; loading predictions and saving files are excluded.


In [9]:
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

config = {
    "schema_version": 1,
    "experiment": "error_before_rounding",
    "shrinkage_order": "before_rounding", "alpha": 0.0,
    "method": METHOD, "model_class": MODEL_CLASS, "prediction_method": PRED_METHOD,
    "need_threshold": NEED_THRESHOLD, "need_column": NEED_COL,
    "expected_need_column": EXP_NEED_COL, "group_column": GROUP_COL,
    "rashomon_epsilon": RASHOMON_EPS, "allocation_epsilon": ALLOCATION_EPS,
    "n_models": N_MODELS, "sample_size": SAMPLE_SIZE, "selection_rate": SELECTION_RATE,
    "k": reference_allocator.k, "iters_per_seed": ITERS_PER_SEED, "seeds": seeds,
    "prediction_path": str(PREDICTION_PATH), "prediction_sha256": file_hash(PREDICTION_PATH),
    "raw_data_path": str(RAW_DATA_PATH), "raw_data_sha256": file_hash(RAW_DATA_PATH),
    "allocation_code_sha256": file_hash(ROOT / "src/allocations.py"),
}
config_text = json.dumps(config, sort_keys=True, indent=2)
config_id = hashlib.sha256(config_text.encode()).hexdigest()[:12]
OUTPUT_DIR = RESULTS_ROOT / f"size-{SAMPLE_SIZE}_models-{N_MODELS}_iters-{ITERS_PER_SEED}_{config_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "config.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != config:
        raise ValueError("The output directory contains a different experiment configuration.")
else:
    manifest_path.write_text(config_text + "\n")

def run_paths(beta, step):
    suffix = f"beta-{float(beta)}_delta-{float(step)}"
    return {
        "allocations": OUTPUT_DIR / f"lda_exp_{suffix}.csv",
        "people": OUTPUT_DIR / f"people_{suffix}.csv",
        "diagnostics": OUTPUT_DIR / f"diagnostics_{suffix}.csv",
        "metadata": OUTPUT_DIR / f"run_{suffix}.json",
    }

def cohort_identity(people):
    keys = set(zip(people["seed"], people["iteration"]))
    expected = {(seed, i) for seed in seeds for i in range(ITERS_PER_SEED)}
    if keys != expected or len(people) != len(expected):
        raise ValueError("Cohort coverage differs from the requested configuration.")
    identity = []
    for row in people.sort_values(["seed", "iteration"]).itertuples(index=False):
        if len(row.people) != SAMPLE_SIZE or len(set(row.people)) != SAMPLE_SIZE:
            raise ValueError("A saved cohort has missing or duplicate people.")
        identity.append((int(row.seed), int(row.iteration), tuple(row.people)))
    return tuple(identity)

def validate_diagnostics(diagnostics, beta, step):
    keys = set(zip(diagnostics["seed"], diagnostics["iteration"], diagnostics["model"]))
    if keys != expected_diagnostic_keys or len(diagnostics) != len(expected_diagnostic_keys):
        raise ValueError("Diagnostic rows do not cover the same eligible models and cohorts.")
    if (not diagnostics["beta"].eq(beta).all()
            or not diagnostics["alpha"].eq(0).all()
            or not np.allclose(diagnostics["need_step"], step, rtol=0, atol=1e-12)
            or not diagnostics["shrinkage_order"].eq("before_rounding").all()):
        raise ValueError("Diagnostic experiment settings do not match this run.")
    for metric in ("mae", "rmse"):
        if not np.allclose(diagnostics[f"before_rounding_{metric}"],
                           beta * diagnostics[f"prediction_{metric}"], rtol=1e-10, atol=1e-12):
            raise ValueError("Pre-rounding errors do not scale with beta as expected.")

print("Results directory:", OUTPUT_DIR)


Results directory: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/results/allocations/error_before_rounding/size-100_models-100_iters-5_7807aca5c013


In [10]:
run_records = []
diagnostic_frames = []
reference_cohorts = None

for beta in BETAS:
    for step in NEED_STEPS:
        paths = run_paths(beta, step)
        if SKIP_EXISTING and paths["metadata"].exists():
            metadata = json.loads(paths["metadata"].read_text())
            if (metadata["config_id"] != config_id or metadata["beta"] != beta
                    or metadata["need_step"] != step):
                raise ValueError(f"Mismatched run metadata: {paths['metadata']}")
            for name in ("allocations", "people", "diagnostics"):
                if not paths[name].exists() or file_hash(paths[name]) != metadata["sha256"][name]:
                    raise ValueError(f"Incomplete or changed saved result: {paths[name]}")
            people = pd.read_csv(paths["people"], converters={"people": literal_eval})
            diagnostics = pd.read_csv(paths["diagnostics"])
            print(f"Using saved beta={beta:g}, delta={step:g}")
        else:
            print(f"Running beta={beta:g}, delta={step:g}")
            allocator = make_allocator()
            started = perf_counter()
            opt, true_dp, allocations, people = allocator.get_allocations(
                exp_need_col=EXP_NEED_COL, algorithms="LDA-exp",
                alpha=0.0, beta=beta,
                shrinkage_order="before_rounding", need_step=step,
            )
            elapsed = perf_counter() - started
            if not opt.empty or not true_dp.empty:
                raise AssertionError("This sweep should only return LDA-exp allocations.")
            diagnostics = pd.DataFrame(allocator.need_diagnostics)
            validate_diagnostics(diagnostics, beta, step)
            identity = cohort_identity(people)
            if reference_cohorts is not None and identity != reference_cohorts:
                raise ValueError("Cohorts changed between beta/grid combinations.")

            # Remove the completion marker before replacing any existing run files.
            paths["metadata"].unlink(missing_ok=True)
            allocations.to_csv(paths["allocations"], index=False)
            people.to_csv(paths["people"], index=False)
            diagnostics.to_csv(paths["diagnostics"], index=False)
            metadata = {
                "config_id": config_id, "algorithm": "LDA-exp", "shrinkage_order": "before_rounding",
                "alpha": 0.0, "beta": float(beta), "need_step": float(step),
                "need_scale": int(diagnostics["need_scale"].iloc[0]),
                "elapsed_seconds": elapsed, "n_cohorts": len(people),
                "n_model_cohort_allocations": len(diagnostics),
                "n_saved_allocation_rows": len(allocations),
                "sha256": {name: file_hash(paths[name]) for name in ("allocations", "people", "diagnostics")},
            }
            temp_metadata = paths["metadata"].with_suffix(".json.tmp")
            temp_metadata.write_text(json.dumps(metadata, indent=2) + "\n")
            temp_metadata.replace(paths["metadata"])
            print(f"Saved in {elapsed:.1f} seconds")

        identity = cohort_identity(people)
        if reference_cohorts is None:
            reference_cohorts = identity
        elif identity != reference_cohorts:
            raise ValueError("Cohorts changed between beta/grid combinations.")
        validate_diagnostics(diagnostics, beta, step)
        run_records.append({key: value for key, value in metadata.items() if key != "sha256"})
        diagnostic_frames.append(diagnostics)

run_summary = pd.DataFrame(run_records).sort_values(["beta", "need_step"], ascending=[True, False])
all_diagnostics = pd.concat(diagnostic_frames, ignore_index=True)
run_summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)
display(run_summary[["beta", "need_step", "elapsed_seconds", "n_cohorts", "n_model_cohort_allocations"]])


Running beta=0, delta=1


[Seed 4]: 100%|██████████| 5/5 [00:41<00:00,  8.32s/it]


Saved in 181.6 seconds
Running beta=0, delta=0.5


[Seed 4]: 100%|██████████| 5/5 [00:41<00:00,  8.34s/it]


Saved in 181.6 seconds
Running beta=0, delta=0.2


...

[Seed 4]: 100%|██████████| 5/5 [01:08<00:00, 13.70s/it]


Saved in 306.0 seconds
Running beta=1, delta=0.2


[Seed 4]: 100%|██████████| 5/5 [02:41<00:00, 32.33s/it]

Saved in 724.4 seconds


,beta,need_step,elapsed_seconds,n_cohorts,n_model_cohort_allocations
0,0.00,1.0,181.554013,25,2490
1,0.00,0.5,181.613247,25,2490
2,0.00,0.2,181.808516,25,2490
3,0.25,1.0,171.738649,25,2490
4,0.25,0.5,321.206735,25,2490
5,0.25,0.2,760.395542,25,2490
6,0.50,1.0,165.337208,25,2490
7,0.50,0.5,314.390092,25,2490
8,0.50,0.2,748.015960,25,2490
9,0.75,1.0,161.990247,25,2490


In [11]:
# Mean error metrics across model/cohort rows, not allocation disparity.
error_columns = [
    "prediction_mae", "before_rounding_mae", "rounding_mae", "after_rounding_mae",
    "prediction_rmse", "before_rounding_rmse", "rounding_rmse", "after_rounding_rmse",
]
error_summary = all_diagnostics.groupby(["beta", "need_step"])[error_columns].mean()
error_summary.to_csv(OUTPUT_DIR / "error_summary.csv")
display(error_summary)
print("Saved allocations, cohorts, diagnostics, and runtime metadata to:", OUTPUT_DIR)


prediction_mae  before_rounding_mae  rounding_mae  \
beta need_step                                                      
0.00 0.2              0.694445             0.000000      0.000000   
     0.5              0.694445             0.000000      0.000000   
     1.0              0.694445             0.000000      0.000000   
0.25 0.2              0.694445             0.173611      0.040041   
     0.5              0.694445             0.173611      0.083632   
     1.0              0.694445             0.173611      0.126436   
0.50 0.2              0.694445             0.347222      0.051776   
     0.5              0.694445             0.347222      0.088380   
     1.0              0.694445             0.347222      0.167263   
0.75 0.2              0.694445             0.520834      0.056198   
     0.5              0.694445             0.520834      0.112096   
     1.0              0.694445             0.520834      0.178971   
1.00 0.2              0.694445             0.694445      0.056242   
     0.5              0.694445             0.694445      0.120978   
     1.0              0.694445             0.694445      0.176761   

                after_rounding_mae  prediction_rmse  before_rounding_rmse  \
beta need_step                                                              
0.00 0.2                  0.000000         1.203934              0.000000   
     0.5                  0.000000         1.203934              0.000000   
     1.0                  0.000000         1.203934              0.000000   
0.25 0.2                  0.153317         1.203934              0.300984   
     0.5                  0.134759         1.203934              0.300984   
     1.0                  0.093643         1.203934              0.300984   
0.50 0.2                  0.327872         1.203934              0.601967   
     0.5                  0.313687         1.203934              0.601967   
     1.0                  0.269518         1.203934              0.601967   
0.75 0.2                  0.508164         1.203934              0.902951   
     0.5                  0.470028         1.203934              0.902951   
     1.0                  0.475414         1.203934              0.902951   
1.00 0.2                  0.692522         1.203934              1.203934   
     0.5                  0.651972         1.203934              1.203934   
     1.0                  0.627373         1.203934              1.203934   

                rounding_rmse  after_rounding_rmse  
beta need_step                                      
0.00 0.2             0.000000             0.000000  
     0.5             0.000000             0.000000  
     1.0             0.000000             0.000000  
0.25 0.2             0.047428             0.299377  
     0.5             0.112827             0.309163  
     1.0             0.184225             0.314889  
0.50 0.2             0.057955             0.602816  
     0.5             0.107735             0.611391  
     1.0             0.225655             0.618325  
0.75 0.2             0.062728             0.903616  
     0.5             0.128970             0.904630  
     1.0             0.223549             0.933211  
1.00 0.2             0.063203             1.205673  
     0.5             0.136666             1.212756  
     1.0             0.215470             1.222782

Saved allocations, cohorts, diagnostics, and runtime metadata to: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/results/allocations/error_before_rounding/size-100_models-100_iters-5_7807aca5c013
